# Risk Methods: Focused Examples

Short examples showing the inputs and outputs behind the risk-model, attribution, and stress-testing modules.

In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd

project_root = Path.cwd().resolve()
if project_root.name == 'notebooks':
    project_root = project_root.parent
sys.path.insert(0, str(project_root / 'src'))

from portfolio_risk.config import CONFIDENCE_LEVEL, RISK_LIMITS, WEIGHTS
from portfolio_risk.factor_risk import default_factor_exposures, factor_return_proxies
from portfolio_risk.market_data import generate_market_data
from portfolio_risk.pnl_attribution import factor_pnl_explain, risk_attribution
from portfolio_risk.portfolio import portfolio_returns
from portfolio_risk.risk_models import compare_risk_models
from portfolio_risk.stress_testing import default_scenarios, reverse_stress, run_stress_tests

pd.options.display.float_format = '{:,.2%}'.format
asset_returns = generate_market_data()
portfolio_pnl = portfolio_returns(asset_returns, WEIGHTS)

## 1. Compare four one-day risk estimates

In [ ]:
risk_models = compare_risk_models(portfolio_pnl, CONFIDENCE_LEVEL)
display(risk_models)

# Small, readable checks on the output table.
assert list(risk_models.index) == ['Historical', 'Normal', 'EWMA', 'Monte Carlo']
assert (risk_models['Expected shortfall'] >= risk_models['VaR']).all()

## 2. Explain daily P&L and identify risk drivers

In [ ]:
exposures = default_factor_exposures(asset_returns.columns)
pnl_explain, factor_contributions = factor_pnl_explain(
    asset_returns, WEIGHTS, exposures, factor_return_proxies(asset_returns)
)

display(pnl_explain.tail())
display(risk_attribution(asset_returns, WEIGHTS))
assert np.allclose(pnl_explain['actual_pnl'], pnl_explain['factor_pnl'] + pnl_explain['residual_pnl'])

## 3. Stress a portfolio and scale shocks to a limit

In [ ]:
stresses = run_stress_tests(WEIGHTS, default_scenarios())
reverse = reverse_stress(WEIGHTS, default_scenarios(), RISK_LIMITS['max_drawdown'])

display(stresses)
display(reverse)
assert (reverse['shock_multiplier'] > 0).all()